In [ ]:
# 1. TensorRT-LLM 설치 (x86 Colab용)
!pip install tensorrt_llm -U --pre --extra-index-url https://pypi.nvidia.com
!pip install --upgrade transformers

# 2. TensorRT-LLM 소스코드 다운로드 (스크립트 사용 목적)
# ※ 주의: Jetson에 설치될 버전과 호환성을 위해 최신 안정 버전을 사용하는 것이 좋습니다.
!git clone https://github.com/NVIDIA/TensorRT-LLM.git
!cd TensorRT-LLM && git submodule update --init --recursive

# 3. 필요한 라이브러리 설치
!pip install -r TensorRT-LLM/examples/llama/requirements.txt

In [ ]:
# git-lfs 설치
!git lfs install

# 모델 다운로드 (Meta-Llama-3-8B-Instruct)
# 시간이 좀 걸립니다. 여유를 가지세요.
!git clone https://huggingface.co/meta-llama/Meta-Llama-3-8B-Instruct model_input

In [ ]:
import sys
import os

# 스크립트 위치로 이동
os.chdir('/content/TensorRT-LLM/examples/llama')

# 변환 및 양자화 실행
# --use_weight_only: 가중치만 양자화 (메모리 절약)
# --weight_only_precision int4_awq: INT4 정밀도 사용
!python convert_checkpoint.py --model_dir /content/model_input \
                              --output_dir /content/tllm_checkpoint_int4 \
                              --dtype float16 \
                              --use_weight_only \
                              --weight_only_precision int4_awq

In [ ]:
# 폴더 압축
!tar -czvf tllm_checkpoint_int4.tar.gz /content/tllm_checkpoint_int4

# (선택) 구글 드라이브로 복사
from google.colab import drive
drive.mount('/content/drive')
!cp tllm_checkpoint_int4.tar.gz /content/drive/MyDrive/

Jetson Orin에서

In [ ]:
# Jetson 터미널에서
mkdir ~/llm_project
mv tllm_checkpoint_int4.tar.gz ~/llm_project
cd ~/llm_project
tar -xzvf tllm_checkpoint_int4.tar.gz

In [ ]:
# Jetson 터미널에서 실행
# dustynv의 pre-built 이미지가 사용하기 가장 편합니다.
sudo docker run -it --rm --runtime nvidia --network host \
    -v ~/llm_project:/data \
    dustynv/tensorrt_llm:r36.2.0 \
    bash

# (참고: 이미지가 없으면 자동으로 다운로드됩니다. 시간이 꽤 걸립니다.)
# 컨테이너 내부로 진입하게 됩니다.

In [ ]:
# 컨테이너 내부에서 실행
# trtllm-build 명령어 사용

trtllm-build --checkpoint_dir /data/tllm_checkpoint_int4 \
             --output_dir /data/llama3_int4_engine \
             --gemm_plugin float16 \
             --max_batch_size 1 \
             --max_input_len 2048 \
             --max_output_len 512

Jetson에서 실행

In [ ]:
# Python 스크립트로 테스트
# TensorRT-LLM 예제 폴더가 컨테이너 내 어딘가에 있거나, 직접 간단한 스크립트를 작성합니다.

# 보통 컨테이너 내 /opt/TensorRT-LLM/examples/run.py 같은 경로에 있습니다.
python3 /opt/TensorRT-LLM/examples/run.py \
    --engine_dir /data/llama3_int4_engine \
    --tokenizer_dir /data/tllm_checkpoint_int4 \
    --max_output_len 100 \
    --input_text "Hello, tell me about Jetson Orin Nano."